# Importing Libraries

In [ ]:
import torch
from transformers import DetrImageProcessor, DetrForObjectDetection
import os
from PIL import Image
import numpy as np
from bytetracker import BYTETracker
from io import BytesIO

# Checking for CUDA

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Loading an Example Object Detection Model

In [ ]:
try:
    processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
    model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(DEVICE)
    model.eval()
    print("DETR model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure you have an internet connection to download the model.")

# Checking the existence of the dataset

In [ ]:
sequence_folder = 'MOT17/train/MOT17-04-DPM/img1'

if not os.path.exists(sequence_folder):
    print(f"Error: Dataset folder not found at '{sequence_folder}'.")
    print("Please ensure the 'MOT17' directory is in the same folder as this script.")

In [ ]:
try:
    image_files = sorted([os.path.join(sequence_folder, f) for f in os.listdir(sequence_folder) if f.endswith('.jpg')])[:60]
    if not image_files:
        print(f"No '.jpg' images found in {sequence_folder}. Please check the dataset structure.")
except Exception as e:
    print(f"Error reading image files from '{sequence_folder}': {e}")

# Initializing ByteTrack

In [ ]:
class ByteTrackArgs:
    track_thresh = 0.5
    track_buffer = 30
    match_thresh = 0.8
    mot20 = False

In [ ]:
args = ByteTrackArgs()
tracker = BYTETracker(**vars(args))
all_tracked_objects = []

# Running the Sample Pipeline

In [ ]:
i = 0
frame_idx = i
image_path = image_files[i]

In [ ]:
for frame_idx, image_path in enumerate(image_files):
try:
    image = Image.open(image_path).convert("RGB")
    # --- Object Detection (DETR) ---
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.7)[0]
    # --- Filtering and Formatting for ByteTrack ---
    person_detections_tensors = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        # DETR uses COCO labels; 'person' is class 1
        if label.item() == 1:
            # Create a single tensor for this detection: [x1, y1, x2, y2, score, label]
            # Note: We unsqueeze score and label to make them concatenatable.
            detection_tensor = torch.cat([
                box,
                score.unsqueeze(0),
                label.float().unsqueeze(0) # Convert label to float for consistency
            ])
            person_detections_tensors.append(detection_tensor)
    
    if len(person_detections_tensors) > 0:
        # Stack the list of tensors into a single 2D tensor
        detections_tensor = torch.stack(person_detections_tensors)
        # --- Object Tracking (ByteTrack) ---
        # Pass the PyTorch TENSOR directly to the update method
        online_targets = tracker.update(detections_tensor, [image.height, image.width])
        for t in online_targets:
            # t is a numpy array: [x1, y1, x2, y2, track_id, score, class_id]
            # Get the bounding box (top-left-bottom-right)
            tlbr = t[:4]
            # Get the track ID
            tid = int(t[4])
            # Calculate tlwh (top-left-width-height) from tlbr
            tlwh = [
                tlbr[0],                 # Top-left x
                tlbr[1],                 # Top-left y
                tlbr[2] - tlbr[0],       # Width
                tlbr[3] - tlbr[1]        # Height
            ]
            all_tracked_objects.append({
                "frame_id": frame_idx + 1,
                "track_id": tid,
                "bbox": [int(coord) for coord in tlwh]
            })
except Exception as e:
    print(f"Error processing frame {frame_idx + 1} ({image_path}): {e}")
    continue

# Results

In [ ]:
print("\n" + "="*30)
print("--- Tracking Results (Sample) ---")
print("="*30)
if all_tracked_objects:
    all_tracked_objects.sort(key=lambda x: (x['frame_id'], x['track_id']))
    current_frame = -1
    # Print the first 20 results for a concise summary
    for obj in all_tracked_objects[:20]:
        if obj['frame_id'] != current_frame:
            print(f"\n[Frame {obj['frame_id']}]")
            current_frame = obj['frame_id']
        print(f"  - Track ID: {obj['track_id']}, Bounding Box: [x={obj['bbox'][0]}, y={obj['bbox'][1]}, w={obj['bbox'][2]}, h={obj['bbox'][3]}]")
else:
    print("No objects were successfully tracked.")